In [0]:
%pip install --upgrade --quiet scikit-learn
dbutils.library.restartPython()

In [0]:
# Define job inputs
dbutils.widgets.text("catalog_name", "main", "UC catalog name")
dbutils.widgets.text("schema_name", "default", "UC schema name")
dbutils.widgets.text("num_training_rows", "100", "Rows of data")
dbutils.widgets.text("num_training_columns", "1000", "Number of features")
dbutils.widgets.text("num_labels", "2", "Number labels in target column")
dbutils.widgets.text("registered_model_name", "dummy_data_classifier", "Registered model name")

# Get parameter values (will override widget defaults if run by job)
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
num_training_rows = int(dbutils.widgets.get("num_training_rows"))
num_training_columns = int(dbutils.widgets.get("num_training_columns"))
num_labels = int(dbutils.widgets.get("num_labels"))
registered_model_name = dbutils.widgets.get("registered_model_name")

# Generated values
table = f"synthetic_data_{num_training_rows}_rows_{num_training_columns}_columns_{num_labels}_labels"
full_model_name = f"{catalog_name}.{schema_name}.{registered_model_name}"
label_column_name="target"

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd

# Load synthetic data from Spark table
df = spark.table(f"{catalog_name}.{schema_name}.{table}")

# Convert to Pandas DataFrame for sklearn
pdf = df.toPandas()

# Split features and target
X = pdf.drop(columns=[label_column_name])
y = pdf[label_column_name]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train logistic regression classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)
report = classification_report(y_test, y_pred, output_dict=True)

# Display classification report as a Spark DataFrame
report_df = pd.DataFrame(report).transpose()
display(spark.createDataFrame(report_df.reset_index()))

In [0]:
# Register model
import mlflow

with mlflow.start_run() as run:
    result = mlflow.sklearn.log_model(clf, "model", 
                             registered_model_name=full_model_name,
                              signature=mlflow.models.signature.infer_signature(X, y))
    mlflow.log_params({
        "num_training_rows": num_training_rows,
        "num_training_columns": num_training_columns,
        "num_labels": num_labels
    })
    mlflow.log_metrics({k: v["f1-score"] for k, v in report.items() if isinstance(v, dict)})

In [0]:
from mlflow import MlflowClient

client = MlflowClient()
try:
  # look for existing Champion. If exists, then set newly registered model version as Challenger. If not then set as Champion
  champion_model = client.get_model_version_by_alias(full_model_name, "Champion")
  client.set_registered_model_alias(
    name=full_model_name,
    alias="Challenger",
    version=result.registered_model_version
  )
  print(f"Champion found. Newest version of the model set as Challenger.")
except:
  print(f"No Champion found. Accept the model as Champion, as it's the first one.")
  client.set_registered_model_alias(
    name=full_model_name,
    alias="Champion",
    version=result.registered_model_version
  )